<a href="https://colab.research.google.com/github/Bpatnaik470/Bpatnaik470/blob/main/Copy_of_Capstone_session_8_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task: Cluster the credit card users into different groups to find any meaningful patterns. Use
Principal Component Analysis (PCA) to reduce the dimension of the feature space and then
use the K-means algorithm to find clusters. Import relevant Python libraries.
• Load dataset (CC GENERAL.csv, the name of the file when downloaded from Kaggle.)

Task: Study the various Recommendation Techniques for recommending movies using
movies.csv, ratings.csv datasets

Load movies.csv and ratings.csv dataset

In [ ]:
import pandas as pd

# Load the datasets
movies_df = pd.read_csv('/content/movies.csv')
ratings_df = pd.read_csv('/content/ratings.csv')

print('Movies Dataset:')
display(movies_df.head())

print('\nRatings Dataset:')
display(ratings_df.head())

Movies Dataset:


,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy



Ratings Dataset:


,userId,movieId,rating,timestamp
0,1,1,4.0,964982703
1,1,3,4.0,964981247
2,1,6,4.0,964982224
3,1,47,5.0,964983815
4,1,50,5.0,964982931


In [ ]:
print('Movies Info:')
movies_df.info()
print('\nRatings Info:')
ratings_df.info()

Movies Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9742 entries, 0 to 9741
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   movieId  9742 non-null   int64 
 1   title    9742 non-null   object
 2   genres   9742 non-null   object
dtypes: int64(1), object(2)
memory usage: 228.5+ KB

Ratings Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100836 entries, 0 to 100835
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100836 non-null  int64  
 1   movieId    100836 non-null  int64  
 2   rating     100836 non-null  float64
 3   timestamp  100836 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


In [ ]:
# Merge datasets on movieId
movie_ratings_df = pd.merge(movies_df, ratings_df, on='movieId')

print('Merged Dataframe Shape:', movie_ratings_df.shape)
display(movie_ratings_df.head())

Merged Dataframe Shape: (100836, 6)


,movieId,title,genres,userId,rating,timestamp
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,1,4.0,964982703
1,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,5,4.0,847434962
2,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,7,4.5,1106635946
3,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,15,2.5,1510577970
4,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy,17,4.5,1305696483


Merge both data frames on movieid

Create User-Item Matrix (Hint: Use pandas pivot_table method with index = 'userId',
columns = 'title', values = 'rating' )

In [ ]:
# Create the User-Item Matrix
user_item_matrix = movie_ratings_df.pivot_table(index='userId', columns='title', values='rating')

print('User-Item Matrix Shape:', user_item_matrix.shape)
display(user_item_matrix.head())

User-Item Matrix Shape: (610, 9719)


title,'71 (2014),'Hellboy': The Seeds of Creation (2004),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),'Tis the Season for Love (2015),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),*batteries not included (1987),...,Zulu (2013),[REC] (2007),[REC]² (2009),[REC]³ 3 Génesis (2012),anohana: The Flower We Saw That Day - The Movie (2013),eXistenZ (1999),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.0,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Perform User-based Collaborative Filtering

In [ ]:
# Fill NaNs with the mean of each row (user's mean rating)
user_item_matrix_filled = user_item_matrix.apply(lambda row: row.fillna(row.mean()), axis=1)

# Compute Pearson correlation between users
# We transpose the matrix because corr() computes correlations between columns (users)
user_correlation_matrix = user_item_matrix_filled.T.corr(method='pearson')

print('User Correlation Matrix Shape:', user_correlation_matrix.shape)
display(user_correlation_matrix.head())

User Correlation Matrix Shape: (610, 610)


userId,1,2,3,4,5,6,7,8,9,10,...,601,602,603,604,605,606,607,608,609,610
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,1.264516e-03,5.525772e-04,0.048419,0.021847,-0.045497,-6.199672e-03,0.047013,1.950985e-02,-8.754088e-03,...,0.018127,-0.017172,-0.015221,-3.705875e-02,-2.912138e-02,0.012016,0.055261,0.075224,-2.571255e-02,0.010932
2,0.001265,1.000000e+00,-4.975816e-17,-0.017164,0.021796,-0.021051,-1.111357e-02,-0.048085,7.652805e-16,3.011629e-03,...,-0.050551,-0.031581,-0.001688,-1.265569e-15,-6.430371e-16,0.006226,-0.020504,-0.006001,-6.009100e-02,0.024999
3,0.000553,-4.975816e-17,1.000000e+00,-0.011260,-0.031539,0.004800,-2.587070e-16,-0.032471,-4.812885e-16,3.774064e-16,...,-0.004904,-0.016117,0.017749,-8.106536e-16,-1.430628e-03,-0.037289,-0.007789,-0.013001,-1.168878e-16,0.019550
4,0.048419,-1.716402e-02,-1.125978e-02,1.000000,-0.029620,0.013956,5.809139e-02,0.002065,-5.873603e-03,5.159032e-02,...,-0.037687,0.063122,0.027640,-1.378212e-02,4.003747e-02,0.020590,0.014628,-0.037569,-1.788358e-02,-0.000995
5,0.021847,2.179571e-02,-3.153892e-02,-0.029620,1.000000,0.009111,1.011715e-02,-0.012284,7.750436e-16,-3.316512e-02,...,0.015964,0.012427,0.027076,1.246135e-02,-3.627206e-02,0.026319,0.031896,-0.001751,9.382892e-02,-0.000278


In [ ]:
# Extract correlations for User 1
user_1_similarity = user_correlation_matrix[1]

# Sort the similarities in descending order (excluding User 1's correlation with themselves)
user_1_similarity = user_1_similarity.sort_values(ascending=False).drop(1)

print('Top 10 users most similar to User 1:')
display(user_1_similarity.head(10))

Top 10 users most similar to User 1:


,1
userId,
301,0.124799
597,0.102631
414,0.101348
477,0.099217
57,0.099070
369,0.098295
206,0.096852
535,0.096493
590,0.095191


Sort the User 1 correlation in the descending order

In [ ]:
# Sort the User 1 correlation in descending order
sorted_user_1_similarity = user_1_similarity.sort_values(ascending=False)

print("User 1 Similarity Scores (Sorted Descending):")
display(sorted_user_1_similarity.head(20))

User 1 Similarity Scores (Sorted Descending):


,1
userId,
301,0.124799
597,0.102631
414,0.101348
477,0.099217
57,0.099070
369,0.098295
206,0.096852
535,0.096493
590,0.095191


In [ ]:
# Drop NaN values from the User 1 correlation series
user_1_similarity = user_1_similarity.dropna()

# Alternatively, if you want to drop NaNs from the entire matrix
# user_correlation_matrix = user_correlation_matrix.dropna(how='all', axis=0).dropna(how='all', axis=1)

print('Number of valid user correlations remaining for User 1:', len(user_1_similarity))
display(user_1_similarity.head())

Number of valid user correlations remaining for User 1: 608


,1
userId,
301,0.124799
597,0.102631
414,0.101348
477,0.099217
57,0.099070


In [ ]:
# Select the top 50 most similar users
top_50_users = user_1_similarity.head(50)

print('Top 50 Users most similar to User 1:')
display(top_50_users)

Top 50 Users most similar to User 1:


,1
userId,
301,0.124799
597,0.102631
414,0.101348
477,0.099217
57,0.099070
369,0.098295
206,0.096852
535,0.096493
590,0.095191


In [ ]:
# Get the movie title for movieId 32
movie_title_32 = movies_df[movies_df['movieId'] == 32]['title'].values[0]

# Get the ratings for this movie from the top 50 similar users
top_50_user_ratings = user_item_matrix.loc[top_50_users.index, movie_title_32]

# Identify users who have actually rated the movie (non-NaN)
valid_ratings_mask = top_50_user_ratings.notna()
valid_ratings = top_50_user_ratings[valid_ratings_mask]
valid_similarities = top_50_users[valid_ratings_mask]

if not valid_ratings.empty:
    # Calculate weighted average: sum(rating * similarity) / sum(similarities)
    predicted_rating = (valid_ratings * valid_similarities).sum() / valid_similarities.sum()
    print(f"Predicted rating for User 1 on movie '{movie_title_32}' (ID: 32): {predicted_rating:.2f}")
    print(f"Based on {len(valid_ratings)} ratings from the top 50 similar users.")
else:
    print(f"None of the top 50 similar users have rated '{movie_title_32}'.")

Predicted rating for User 1 on movie 'Twelve Monkeys (a.k.a. 12 Monkeys) (1995)' (ID: 32): 4.14
Based on 30 ratings from the top 50 similar users.
